# 1. 环境准备

## 1.1 python 环境准备

请运行以下代码完成 python 环境的安装：

In [ ]:
! pip install gradio==6.2.0 openai==2.11.0

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 大模型调用函数准备

若想了解更多关于函数部分的内容，请查阅 **2.1 函数** 部分内容。

In [ ]:
import os
from openai import OpenAI

def llm_response(content):
  client = OpenAI(
  api_key=os.environ.get("OPENAI_API_KEY"), 
  base_url="https://aistudio.baidu.com/llm/lmapi/v3", 
  )

  chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'},
    {'role': 'user', 'content': content}
  ],
  model="ernie-3.5-8k",
  )

  return chat_completion.choices[0].message.content

## 1.4 Gradio Interface 简介

在 Gradio 中提供了四种快速构建原型的方式：
- **Interface** —— Gradio 最简单的“黑箱模型”界面（关键参数：函数 + 输入组件 + 输出组件）
- **ChatInterface** —— 基于 Interface 改造的，专为对话式模型设计的界面（大模型时代下快速对话窗口创建）
- **TabbedInterface** —— 多个 Interface 的“标签页包装器”（可组合 ChatInterface 和 Interface 页面）
- **Blocks** —— Gradio 最强的“网页搭建框架”（实现自由的组件化页面构建方式）

> **温馨提示：**  ：往后所有的 gradio 相关内容请创建 xxx.gradio.py 文件后在终端进行运行，BML 中的 Juypter Notebook 并不支持直接展示。

# 2. Interface 界面创建

## 2.1 快速创建

假如我们想要快速创建，可以先创建一个简单的函数并结合 `gr.Interface()` 完成。

In [ ]:
import gradio as gr

def greet(name):
  return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="textbox", outputs="textbox")

if __name__ == "__main__":
  demo.launch()

最后两部分代码的具体含义是：

- **demo.launch()**
调用 `launch()` 方法会启动一个本地 Web 服务，并在浏览器中打开交互界面。你可以在页面上输入内容，查看函数运行结果。

- **if __name__ == "__main__":**
这是 Python 的标准写法，确保当前脚本只有在**直接运行**时才会启动 Gradio 服务。如果此文件被作为模块导入到其他文件中，`demo.launch()` 不会自动执行，避免意外启动服务。

## 2.2 传入 Interface() 参数

这里我们演示中额外添加的就有 `title` 和 `description` ，其作用就是给页面添加一个标题以及描述性的内容。

In [ ]:
import gradio as gr

def greet(name):
  return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="textbox", outputs="textbox",
          title='测试', description='测试页面')

if __name__ == "__main__":
  demo.launch()

## 2.3 传入组件参数

这里演示 `gr.Textbox()` 我们只输入了 value 和 info 两个参数。

In [ ]:
import gradio as gr

def greet(name):
  return "Hello " + name + "!"

demo = gr.Interface(fn=greet, 
                    inputs=gr.Textbox(value='ljf',info='请写入你的名称'), 
                    outputs=gr.Textbox(), 
                    title='测试', 
                    description='测试页面')

if __name__ == "__main__":
  demo.launch()

除了 [gr.Textbox()](https://www.gradio.app/docs/gradio/textbox) 这个组件以外，其他组件的操作也是非常类似的，我们可以在 [Gradio 官方文档的目录](https://www.gradio.app/docs/gradio/interface) 里找到相关的信息。

## 2.4 多参数输入输出

假如我们的函数输入和输出都是多个组件的话，我们就需要使用列表 `[]` 将多个组件放进去，并且使用逗号 `,` 将其隔开。

In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * intensity

demo = gr.Interface(
    fn=greet,
    inputs=["text", gr.Slider(value=2, minimum=1, maximum=10, step=1)],
    outputs=[gr.Textbox(label="greeting", lines=3)],
    title='测试', 
    description='测试页面')

demo.launch()

# 3. ChatInterface 页面构建

## 3.1 ChatInterface 简介

对于 ChatInterface 而言，其核心只有一件事 —— 实现一个 chat 函数，输入是：
- message：用户这一次发的内容（一般是字符串）
- history：之前的对话历史，是 [{role, content}, ...] 这种 OpenAI 风格字典列表（gradio 内部会自动进行处理）

输出是：
- response：调用大模型后的回复（字符串）

比如：

```python 
def chat_fn(message, history):
    ...
    return response
```

## 3.2 快速创建

假如我们要为前面的 `llm_response` 函数搭配上一个 ChatInterface，我们可以通过以下代码实现：

In [ ]:
import os
from openai import OpenAI
import gradio as gr

def llm_response(content):
  client = OpenAI(
  api_key=os.environ.get("OPENAI_API_KEY"), 
  base_url="https://aistudio.baidu.com/llm/lmapi/v3", 
  )

  chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'},
    {'role': 'user', 'content': content}
  ],
  model="ernie-3.5-8k",
  )

  return chat_completion.choices[0].message.content

def chat_fn(message, history):
  reply = llm_response(message)
  return reply

demo = gr.ChatInterface(
  fn=chat_fn
)

demo.launch()

## 3.3 添加参数

除了最基础的 fn 参数以外，还有其他常用参数可以设置：
- title：标题
- description：对该对话机器的描述
- examples：例子
- cache_examples：预计算 examples，点击时可秒出结果
- save_history：保留聊天历史在内存中（但此时还是没有上下文记忆，只是可以打开多个聊天窗口，上下文记忆将在 langchain 部份讲解）
- multimodal：开启后可以传图片等多模态内容
- editable：是否能够重新编辑文字输入
- chatbot：假如我们想要调整聊天框的尺寸或者设置聊天窗口的默认值，我们可以传入一个 gr.Chatbot 组件进去。 

In [ ]:
import os
from openai import OpenAI
import gradio as gr

def llm_response(content):
  client = OpenAI(
  api_key=os.environ.get("OPENAI_API_KEY"), 
  base_url="https://aistudio.baidu.com/llm/lmapi/v3", 
  )

  chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'},
    {'role': 'user', 'content': content}
  ],
  model="ernie-3.5-8k",
  )

  return chat_completion.choices[0].message.content

def chat_fn(message, history):
  reply = llm_response(message)
  return reply

demo = gr.ChatInterface(
 fn=chat_fn,
 title="AI Studio 小助理",
 description="能够提供开发相关的知识，给开发者提供帮助。",
 examples=[
  "我在使用AI Studio实训平台时遇到了问题，应该如何寻求帮助？",
  "AI Studio实训平台支持哪些编程语言？",
  "如何在AI Studio实训平台上部署我的模型？"],
 cache_examples= False,
 save_history = True,
 multimodal=False,
 editable=True,
 chatbot=gr.Chatbot(
 label="AI Studio 小助理 Chatbot",
 value=[{"role": "user", "content": "请输入问题后开始对话。"}, 
 {"role": "assistant", "content": "你好！我是AI Studio小助理"}],
 min_width = 200))

demo.launch()

# 4. TabbedInterface 页面简介

TabbedInterface 本质上就是传入多个 Interface 的信息，然后通过 Tab 选项卡的形式放到一个页面中而已，并没有太多特殊的内容。

这里我们可以把前面创建的两个 Interface 和 ChatInterface 组合在一起：

In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * intensity

demo1 = gr.Interface(
    fn=greet,
    inputs=["text", gr.Slider(value=2, minimum=1, maximum=10, step=1)],
    outputs=[gr.Textbox(label="greeting", lines=3)],
    title='测试', 
    description='测试页面')

In [ ]:
import os
from openai import OpenAI
import gradio as gr

def llm_response(content):
  client = OpenAI(
  api_key=os.environ.get("OPENAI_API_KEY"), 
  base_url="https://aistudio.baidu.com/llm/lmapi/v3", 
  )

  chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'},
    {'role': 'user', 'content': content}
  ],
  model="ernie-3.5-8k",
  )

  return chat_completion.choices[0].message.content

def chat_fn(message, history):
  reply = llm_response(message)
  return reply

demo2 = gr.ChatInterface(
  fn=chat_fn,
  title="AI Studio 小助理",
  description="能够提供开发相关的知识，给开发者提供搜索帮助建议。",
  examples=[
    "我在使用AI Studio实训平台时遇到了问题，应该如何寻求帮助？",
    "AI Studio实训平台支持哪些编程语言？",
    "如何在AI Studio实训平台上部署我的模型？",
  ],
  cache_examples=False,
  save_history = True,
)

In [ ]:
demo = gr.TabbedInterface(
  interface_list=[demo1, demo2], 
  tab_names=["interface", "chatbot"])

demo.launch()